In [10]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python path: {sys.path}")

Python executable: c:\Users\Anupam\Desktop\LLM\.venv\Scripts\python.exe
Python path: ['C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\python310.zip', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\DLLs', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10\\lib', 'C:\\Users\\Anupam\\.pyenv\\pyenv-win\\versions\\3.10.10', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv', '', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\win32', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\win32\\lib', 'c:\\Users\\Anupam\\Desktop\\LLM\\.venv\\lib\\site-packages\\Pythonwin']


In [1]:
try:
    import trl
    print(f"✅ trl is installed at: {trl.__file__}")
    print(f"trl version: {trl.__version__}")
except ImportError as e:
    print(f"❌ trl import failed: {e}")

c:\Users\Anupam\Desktop\LLM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ trl is installed at: c:\Users\Anupam\Desktop\LLM\.venv\lib\site-packages\trl\__init__.py
trl version: 0.22.2


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.6.0+cu124
True


In [2]:
import os
import torch
from trl import SFTTrainer
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments


Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu126 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0309 21:56:04.657000 39548 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\Anupam\AppData\Local\Temp\ipykernel_39548\4242190646.py:5: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[fla.utils|WARNING]Detected Windows operating system. Triton does not have an official Windows release, thus FLA will not be adapted for Windows, and any potential errors will not be fixed. Please consider using a Linux environment for compatibility.
[fla.utils|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
max_seq_length = 2048
model_name= "unsloth/Qwen3.5-2B"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit = False,     # MoE QLoRA not recommended, dense 27B is fine
    load_in_16bit = True,     # bf16/16-bit LoRA
    full_finetuning = False,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 617/617 [00:04<00:00, 134.20it/s, Materializing param=model.visual.pos_embed.weight]                                 


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    # without dropout and without bias, to speed up training.
    lora_dropout = 0,
    bias = "none",
    # "unsloth" checkpointing is intended for very long context + lower VRAM
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = max_seq_length,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


Prepare the Dataset

In [7]:
dataset = load_dataset("nohurry/Opus-4.6-Reasoning-3000x-filtered", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=3407)

In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'problem', 'thinking', 'solution', 'difficulty', 'category', 'timestamp', 'hash'],
        num_rows: 2093
    })
    test: Dataset({
        features: ['id', 'problem', 'thinking', 'solution', 'difficulty', 'category', 'timestamp', 'hash'],
        num_rows: 233
    })
})

In [14]:
from unsloth import is_bfloat16_supported
from trl import SFTConfig

In [15]:
# Define SFT-specific configuration
sft_config = SFTConfig(
    dataset_text_field="text",  # Moved from SFTTrainer to SFTConfig
    max_length=2048,         # Note: may need to use max_length instead
    packing=True,                # Moved from SFTTrainer to SFTConfig
    dataset_num_proc=2,          # Moved from SFTTrainer to SFTConfig
)

In [19]:
# Training arguments remain the same
training_args = TrainingArguments(
    learning_rate=3e-4,
    lr_scheduler_type="linear",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    warmup_steps=10,
    output_dir="output",
    report_to="mlflow",  # This should work with comet_ml installed [citation:4][citation:8]
    seed=0,
)

In [20]:
# Initialize SFTTrainer with proper parameter placement
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,  # Correct for v0.22.0 [citation:5]
    args=training_args,            # Pass SFTConfig here [citation:9]
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


ValueError: No columns in the dataset match the model's forward method signature: (messages, prompt, completion, images). The following columns have been ignored: [problem, solution, hash, difficulty, thinking, category, id, timestamp]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.